In [5]:
%run ../module7-genai-langchain/initialize_environment.py

Environment initializing completed successfully.


## Summarize messages

In [ ]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import SummarizationMiddleware

llm = create_azure_llm()

agent = create_agent(
    model=llm,
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=llm,
            trigger=("tokens", 100),
            keep=("messages", 1)
        )
    ],
)

Using chat deployment: lums-gpt-4.1-mini
Endpoint: https://aoai-foundry-swc.openai.azure.com/
API Version: 2025-01-01-preview


In [9]:
from langchain.messages import HumanMessage, AIMessage
from pprint import pprint

response = agent.invoke(
    {"messages": [
        HumanMessage(content="What is the capital of the moon?"),
        AIMessage(content="The capital of the moon is Lunapolis."),
        HumanMessage(content="What is the weather in Lunapolis?"),
        AIMessage(content="Skies are clear, with a high of 120C and a low of -100C."),
        HumanMessage(content="How many cheese miners live in Lunapolis?"),
        AIMessage(content="There are 100,000 cheese miners living in Lunapolis."),
        HumanMessage(content="Do you think the cheese miners' union will strike?"),
        AIMessage(content="Yes, because they are unhappy with the new president."),
        HumanMessage(content="If you were Lunapolis' new president how would you respond to the cheese miners' union?"),
        ]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

{'messages': [HumanMessage(content="Here is a summary of the conversation to date:\n\n## SESSION INTENT\nThe user is engaging in a fictional or imaginative inquiry about the moon, specifically focusing on the capital city, its weather, the population of cheese miners, and the possibility of a strike by their union.\n\n## SUMMARY\n- The moon's capital is named Lunapolis.\n- The weather in Lunapolis features extreme temperatures: a high of 120°C and a low of -100°C with clear skies.\n- There are 100,000 cheese miners living in Lunapolis.\n- The cheese miners' union is likely to strike due to dissatisfaction with the new president.\n\n## ARTIFACTS\nNone\n\n## NEXT STEPS\nNone indicated; the conversation focused on answering imaginative questions about Lunapolis and its inhabitants.", additional_kwargs={'lc_source': 'summarization'}, response_metadata={}, id='7a7ea52c-e539-4344-930b-c1f84155f958'),
              HumanMessage(content="If you were Lunapolis' new president how would you respo

In [10]:
print(response["messages"][0].content)

Here is a summary of the conversation to date:

## SESSION INTENT
The user is engaging in a fictional or imaginative inquiry about the moon, specifically focusing on the capital city, its weather, the population of cheese miners, and the possibility of a strike by their union.

## SUMMARY
- The moon's capital is named Lunapolis.
- The weather in Lunapolis features extreme temperatures: a high of 120°C and a low of -100°C with clear skies.
- There are 100,000 cheese miners living in Lunapolis.
- The cheese miners' union is likely to strike due to dissatisfaction with the new president.

## ARTIFACTS
None

## NEXT STEPS
None indicated; the conversation focused on answering imaginative questions about Lunapolis and its inhabitants.


## Trim/delete messages

In [11]:
from typing import Any
from langchain.agents import AgentState
from langchain.messages import RemoveMessage
from langgraph.runtime import Runtime
from langchain.agents.middleware import before_agent
from langchain.messages import ToolMessage

@before_agent
def trim_messages(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """Remove all the tool messages from the state"""
    messages = state["messages"]

    tool_messages = [m for m in messages if isinstance(m, ToolMessage)]
    
    return {"messages": [RemoveMessage(id=m.id) for m in tool_messages]}

In [12]:
llm = create_azure_llm()

agent = create_agent(
    model=llm,
    checkpointer=InMemorySaver(),
    middleware=[trim_messages],
)

Using chat deployment: lums-gpt-4.1-mini
Endpoint: https://aoai-foundry-swc.openai.azure.com/
API Version: 2025-01-01-preview


In [13]:
response = agent.invoke(
    {"messages": [
        HumanMessage(content="My device won't turn on. What should I do?"),
        ToolMessage(content="blorp-x7 initiating diagnostic ping…", tool_call_id="1"),
        AIMessage(content="Is the device plugged in and turned on?"),
        HumanMessage(content="Yes, it's plugged in and turned on."),
        ToolMessage(content="temp=42C voltage=2.9v … greeble complete.", tool_call_id="2"),
        AIMessage(content="Is the device showing any lights or indicators?"),
        HumanMessage(content="What's the temperature of the device?")
        ]},
    {"configurable": {"thread_id": "2"}}
)

pprint(response)

{'messages': [HumanMessage(content="My device won't turn on. What should I do?", additional_kwargs={}, response_metadata={}, id='7ac38ef4-2deb-40a5-bbf0-5435d7229825'),
              AIMessage(content='Is the device plugged in and turned on?', additional_kwargs={}, response_metadata={}, id='db9ae2f5-b892-4499-99b2-71e985ed7a9e', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content="Yes, it's plugged in and turned on.", additional_kwargs={}, response_metadata={}, id='66b15eb1-3feb-47d9-8a1e-91372eb96e56'),
              AIMessage(content='Is the device showing any lights or indicators?', additional_kwargs={}, response_metadata={}, id='0f8424d8-b261-4820-9473-9d8c4690fd55', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content="What's the temperature of the device?", additional_kwargs={}, response_metadata={}, id='10efed90-8a17-444d-967b-b0099c5d30a7'),
              AIMessage(content='Could you please clarify what type of device you’re referring 

In [14]:
print(response["messages"][-1].content)

Could you please clarify what type of device you’re referring to? Also, if you can, check if the device feels unusually hot, cold, or normal to the touch. This information will help me assist you better.
